# DINOv2 Zero-Shot Analysis — Shrouded vs Open Rotor VTOL Patents

**Purpose:** Extract structural embeddings from patent drawings using frozen DINOv2-large (no fine-tuning), reduce dimensionality with PCA, cluster with HDBSCAN, and visualize with UMAP.

**Architecture:** All logic lives in [`src/zero_shot.py`](../src/zero_shot.py). This notebook configures paths/hyperparameters and calls one function per stage.

| Stage | What it does |
|---|---|
| 0 — Paths | Resolve input/output directories via `src.config_loader` |
| 1 — Config | Set hyperparameters; import from `src.zero_shot` |
| 2 — Extract | Batched CLS-token extraction + patent-level mean pooling |
| 3 — Normalize | L2-project embeddings onto the unit hypersphere |
| 3.5 — PCA | Reduce 1024d → 100d (curse of dimensionality fix) |
| 4 — Cluster | HDBSCAN on PCA embeddings; evaluate with Silhouette + Davies-Bouldin |
| 5 — Visualize | UMAP 2D scatter: cluster assignments + ground-truth labels side-by-side |

In [ ]:
# ── Stage 0: Path Configuration ──────────────────────────────────────────────
import sys
from pathlib import Path

# Make pipeline/ importable from notebooks/
sys.path.insert(0, str(Path("..").resolve()))

from src.config_loader import load_config
cfg = load_config()

# Input: 84 processed images in class subfolders (shrouded/ and open_rotor/)
IMAGE_DIR = Path(cfg["paths"]["final"])

# Output: experiment folder / zero_shot_analysis/
ANALYSIS_DIR  = Path(cfg["paths"]["experiment"]) / "zero_shot_analysis"
OUTPUT_DIR    = ANALYSIS_DIR / "outputs"
EMBEDDING_DIR = OUTPUT_DIR / "embeddings"
PLOT_DIR      = OUTPUT_DIR / "plots"

for d in [ANALYSIS_DIR, OUTPUT_DIR, EMBEDDING_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Images  : {IMAGE_DIR}")
print(f"  exists: {IMAGE_DIR.exists()}")
print(f"Outputs : {OUTPUT_DIR}")

In [ ]:
# ── Stage 1: Hyperparameters & Imports ───────────────────────────────────────
import torch
import matplotlib.pyplot as plt
from src.zero_shot import (
    collect_image_paths, patent_id_from_path, category_from_path,
    initialize_dinov2, extract_embeddings,
    l2_normalize, pca_reduce, hdbscan_cluster,
    umap_project, plot_umap_clusters, plot_cluster_gallery,
    safe_save_np, safe_save_df, safe_save_plot,
)

SEED                      = 42
MODEL_NAME                = "facebook/dinov2-large"   # 1024-d CLS token
BATCH_SIZE                = 16
PCA_N_COMPONENTS          = 100    # advisor recommendation: reduce 1024d → 100d
HDBSCAN_MIN_CLUSTER_SIZE  = 5      # ~5% of 84 images; try 4–8
DEVICE                    = torch.device("cuda:1")

print(f"Model : {MODEL_NAME}")
print(f"Device: {DEVICE}  |  Seed: {SEED}")
print(f"PCA   : {PCA_N_COMPONENTS}d  |  HDBSCAN min_cluster_size: {HDBSCAN_MIN_CLUSTER_SIZE}")

In [ ]:
# ── Stage 2: Embedding Extraction ────────────────────────────────────────────
image_paths = collect_image_paths(IMAGE_DIR)
print(f"Found {len(image_paths)} images in {IMAGE_DIR}")

processor, model = initialize_dinov2(MODEL_NAME, DEVICE)

image_emb, img_meta_df, patent_ids, patent_emb = extract_embeddings(
    image_paths, processor, model, DEVICE, BATCH_SIZE
)

safe_save_np(patent_emb,  EMBEDDING_DIR / "patent_embeddings.npy")
safe_save_np(image_emb,   EMBEDDING_DIR / "image_embeddings.npy")
safe_save_df(img_meta_df, EMBEDDING_DIR / "image_metadata.csv")

In [ ]:
# ── Stage 3: L2 Normalization ─────────────────────────────────────────────────
# Projects each 1024-d embedding vector onto the unit hypersphere (length = 1).
# After this, Euclidean distance is equivalent to cosine distance —
# so clustering groups patents by the *direction* (structure) of their embeddings,
# not by their magnitude.
X_norm = l2_normalize(patent_emb)

In [ ]:
# ── Stage 3.5: PCA Dimensionality Reduction ───────────────────────────────────
# DINOv2-large produces 1024-d vectors. In very high dimensions, all pairwise
# distances converge to the same value (curse of dimensionality), making
# distance-based clustering unreliable. PCA keeps the most discriminative
# directions and reduces the space to 100 dimensions (advisor recommendation).
X_pca = pca_reduce(X_norm, n_components=PCA_N_COMPONENTS, seed=SEED)
safe_save_np(X_pca, EMBEDDING_DIR / "patent_embeddings_pca.npy")

In [ ]:
# ── Stage 4: HDBSCAN Clustering ──────────────────────────────────────────────
# Clusters the PCA-reduced embeddings. Unlike DBSCAN, HDBSCAN does not require
# tuning an eps radius — it only needs min_cluster_size, which is intuitive:
# the smallest group of patents that counts as a real cluster.
# Noise points (not assigned to any cluster) get label = -1.
cluster_df, patent_cluster_labels = hdbscan_cluster(
    X_pca, patent_ids, min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE
)
saved = safe_save_df(cluster_df, OUTPUT_DIR / "patent_cluster_assignments_hdbscan.csv")
print(f"Assignments saved: {saved.name}")
print(cluster_df.to_string(index=False))

In [ ]:
# ── Stage 5: UMAP Visualization ──────────────────────────────────────────────
# UMAP reduces PCA-100d embeddings to 2D for plotting.
# Left panel:  HDBSCAN cluster assignments (do the clusters make sense?)
# Right panel: Ground-truth class labels  (do shrouded/open_rotor separate?)
points_2d = umap_project(X_pca, seed=SEED)

# Build per-patent ground-truth label from filename (_SHR_ or _OPN_)
pid_to_gt = {}
for p in image_paths:
    pid = patent_id_from_path(p)
    if pid not in pid_to_gt:
        pid_to_gt[pid] = category_from_path(p)
gt_labels = [pid_to_gt.get(pid, "unknown") for pid in patent_ids]

fig = plot_umap_clusters(
    points_2d,
    patent_cluster_labels,
    patent_ids,
    title_suffix="84 VTOL Patents",
    ground_truth_labels=gt_labels,
)
safe_save_plot(fig, PLOT_DIR / "patent_clusters_umap.png", dpi=220, bbox_inches="tight")
plt.show()

plot_cluster_gallery(cluster_df, patent_ids, image_paths, PLOT_DIR)